In [153]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

import importlib
import helper_fn

importlib.reload(helper_fn)

from prophet import Prophet

import itertools
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [5]:
sellin, sellout = helper_fn.load_sell_data()
sellin, sellout = helper_fn.get_only_fmc(sellin), helper_fn.get_only_fmc(sellout)

In [154]:
sellout[sellout["customer_name"] == 306055.0].groupby("sku_code")[
    "sales_quantity"
].sum().sort_values(ascending=False).head()

sku_code
a0U1t000002PXlqEAG    1269.0
a0U3W000002BYaXUAW    1118.0
a0U1t000002PXloEAG    1015.0
a0U1t000002PXnYEAW     724.0
a0U1t000002PXjoEAG     368.0
Name: sales_quantity, dtype: float64

In [155]:
selected_customer = "0011t000011bAt2AAE"

df_sellin_selected, df_sellout_selected = helper_fn.get_selected_customer_data(
    sellin, sellout, selected_customer, selected_sku="a0U1t000002PXlqEAG"
)

In [165]:
# Prophet expects columns: ds (date) and y (value)
df_prophet = df_sellout_selected.rename(columns={"date": "ds", "sales_quantity": "y"})

# Train / test split — last 7 days as test
TEST_DAYS = 7
df_train = df_prophet.iloc[:-TEST_DAYS].copy()
df_test = df_prophet.iloc[-TEST_DAYS:].copy()

print(
    f"Train: {df_train['ds'].min().date()} → {df_train['ds'].max().date()} ({len(df_train)} days)"
)
print(
    f"Test:  {df_test['ds'].min().date()} → {df_test['ds'].max().date()} ({len(df_test)} days)"
)

Train: 2026-01-03 → 2026-03-24 (81 days)
Test:  2026-03-25 → 2026-03-31 (7 days)


In [166]:
import itertools
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from prophet.diagnostics import cross_validation, performance_metrics

param_grid = {
    "changepoint_prior_scale": [0.01, 0.05, 0.1],
    "seasonality_prior_scale": [0.1, 1.0],
    "holidays_prior_scale": [0.1, 1.0],
    "seasonality_mode": ["additive", "multiplicative"],
}

all_params = [
    dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())
]
print(f"Total combinations: {len(all_params)}")

results = []

for params in all_params:
    m = Prophet(
        weekly_seasonality=True,
        daily_seasonality=False,
        yearly_seasonality=False,
        **params,
    )
    m.add_country_holidays(country_name="FR")
    m.fit(df_train)

    # Predict on validation/test set
    forecast = m.predict(df_test)

    # Merge predictions with actuals
    df_eval = df_test[["ds", "y"]].merge(forecast[["ds", "yhat"]], on="ds", how="left")

    mae = mean_absolute_error(df_eval["y"], df_eval["yhat"])
    rmse = root_mean_squared_error(df_eval["y"], df_eval["yhat"])

    results.append(
        {
            **params,
            "mae": mae,
            "rmse": rmse,
        }
    )

df_results = pd.DataFrame(results).sort_values("mae").reset_index(drop=True)
df_results.head(10)

Total combinations: 24


12:24:31 - cmdstanpy - INFO - Chain [1] start processing


12:24:32 - cmdstanpy - INFO - Chain [1] done processing
12:24:32 - cmdstanpy - INFO - Chain [1] start processing
12:24:33 - cmdstanpy - INFO - Chain [1] done processing
12:24:33 - cmdstanpy - INFO - Chain [1] start processing
12:24:34 - cmdstanpy - INFO - Chain [1] done processing
12:24:34 - cmdstanpy - INFO - Chain [1] start processing
12:24:34 - cmdstanpy - INFO - Chain [1] done processing
12:24:34 - cmdstanpy - INFO - Chain [1] start processing
12:24:35 - cmdstanpy - INFO - Chain [1] done processing
12:24:35 - cmdstanpy - INFO - Chain [1] start processing
12:24:35 - cmdstanpy - INFO - Chain [1] done processing
12:24:35 - cmdstanpy - INFO - Chain [1] start processing
12:24:36 - cmdstanpy - INFO - Chain [1] done processing
12:24:36 - cmdstanpy - INFO - Chain [1] start processing
12:24:36 - cmdstanpy - INFO - Chain [1] done processing
12:24:36 - cmdstanpy - INFO - Chain [1] start processing
12:24:37 - cmdstanpy - INFO - Chain [1] done processing
12:24:37 - cmdstanpy - INFO - Chain [1] 

,changepoint_prior_scale,seasonality_prior_scale,holidays_prior_scale,seasonality_mode,mae,rmse
0,0.10,0.1,1.0,additive,27.053189,29.747995
1,0.10,0.1,0.1,additive,27.053189,29.747995
2,0.10,1.0,1.0,additive,27.078905,29.797485
3,0.10,1.0,0.1,additive,27.078905,29.797485
4,0.10,0.1,0.1,multiplicative,27.359079,30.015294
5,0.10,0.1,1.0,multiplicative,27.359079,30.015294
6,0.10,1.0,1.0,multiplicative,27.414537,30.072956
7,0.10,1.0,0.1,multiplicative,27.414537,30.072956
8,0.01,0.1,0.1,additive,28.870231,32.271205
9,0.01,0.1,1.0,additive,28.870231,32.271205


In [167]:
import plotly.graph_objects as go

best_params = df_results.iloc[0][list(param_grid.keys())].to_dict()
print("Best params:", best_params)

best_model = Prophet(
    weekly_seasonality=True,
    daily_seasonality=False,
    yearly_seasonality=False,
    **best_params,
)
best_model.add_country_holidays(country_name="FR")
best_model.fit(df_train)

df_pred = best_model.predict(df_test[["ds"]])
best_eval = df_test.merge(df_pred[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds")

mae = mean_absolute_error(best_eval["y"], best_eval["yhat"])
rmse = root_mean_squared_error(best_eval["y"], best_eval["yhat"])
print(f"Test MAE: {mae:.2f} | Test RMSE: {rmse:.2f}")

split_date = df_test["ds"].min().strftime("%Y-%m-%d")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.6,
    )
)
fig.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["y"],
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
    )
)
fig.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["yhat"],
        name="Forecast",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dash"),
        marker=dict(size=6),
    )
)
fig.add_trace(
    go.Scatter(
        x=pd.concat([best_eval["ds"], best_eval["ds"][::-1]]),
        y=pd.concat([best_eval["yhat_upper"], best_eval["yhat_lower"][::-1]]),
        fill="toself",
        fillcolor="rgba(220,20,60,0.15)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Confidence interval",
    )
)
fig.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train/Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig.update_layout(
    title=f"Prophet Forecast — {selected_customer}<br><sup>MAE: {mae:.2f} | RMSE: {rmse:.2f}</sup>",
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.show()

12:24:45 - cmdstanpy - INFO - Chain [1] start processing


Best params: {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 0.1, 'holidays_prior_scale': 1.0, 'seasonality_mode': 'additive'}


12:24:45 - cmdstanpy - INFO - Chain [1] done processing


Test MAE: 27.05 | Test RMSE: 29.75


In [168]:
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# ── Extract Prophet in-sample features for train ─────────────────────────
# best_model is already fitted on df_train (from cell d4423dd5)
prophet_train_pred = best_model.predict(df_train[["ds"]])
prophet_test_pred = best_model.predict(df_test[["ds"]])

PROPHET_FEATS = ["trend", "weekly", "yhat", "yhat_lower", "yhat_upper"]
if "holidays" in prophet_train_pred.columns:
    PROPHET_FEATS.append("holidays")


def build_features_t2(prophet_pred_df, ds_series):
    feat = prophet_pred_df[PROPHET_FEATS].copy()
    feat["day_of_week"] = ds_series.dt.dayofweek.values
    feat["is_weekend"] = feat["day_of_week"].isin([5, 6]).astype(int)
    return feat


X_t2_train = build_features_t2(prophet_train_pred, df_train["ds"])
y_t2_train = df_train["y"].values
X_t2_test = build_features_t2(prophet_test_pred, df_test["ds"])
y_t2_test = df_test["y"].values

# ── Optuna tuning (CV on train) ───────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=3)


def objective_t2(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 30),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv.split(X_t2_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_t2_train.iloc[tr_idx], y_t2_train[tr_idx])
        fold_maes.append(
            mean_absolute_error(
                y_t2_train[val_idx], m.predict(X_t2_train.iloc[val_idx])
            )
        )
    return np.mean(fold_maes)


optuna.logging.set_verbosity(optuna.logging.WARNING)
study_t2 = optuna.create_study(direction="minimize")
study_t2.optimize(objective_t2, n_trials=60)
print("Best CV MAE (T2):", round(study_t2.best_value, 3))
print("Best params (T2):", study_t2.best_params)

# ── Refit on full train, evaluate on test ────────────────────────────────
best_t2 = lgb.LGBMRegressor(**study_t2.best_params, verbosity=-1, random_state=42)
best_t2.fit(X_t2_train, y_t2_train)
t2_preds = best_t2.predict(X_t2_test)

t2_mae = mean_absolute_error(y_t2_test, t2_preds)
t2_rmse = root_mean_squared_error(y_t2_test, t2_preds)
print(f"\nTechnique 2 — Prophet + LightGBM")
print(f"  Test MAE : {t2_mae:.2f}  (Prophet alone: {mae:.2f})")
print(f"  Test RMSE: {t2_rmse:.2f}  (Prophet alone: {rmse:.2f})")

# ── Plot ─────────────────────────────────────────────────────────────────
import plotly.graph_objects as go

split_date = df_test["ds"].min().strftime("%Y-%m-%d")
fig_t2 = go.Figure()
fig_t2.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.5,
    )
)
fig_t2.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=y_t2_test,
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
    )
)
fig_t2.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t2_preds,
        name="Prophet + LightGBM",
        mode="lines+markers",
        line=dict(color="darkorange", width=2, dash="dash"),
        marker=dict(size=6),
    )
)
fig_t2.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["yhat"],
        name="Prophet (baseline)",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dot"),
        marker=dict(size=6),
    )
)
fig_t2.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig_t2.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train / Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig_t2.update_layout(
    title=(
        f"Technique 2: Prophet + LightGBM — {selected_customer}<br>"
        f"<sup>T2  MAE: {t2_mae:.2f} | RMSE: {t2_rmse:.2f}"
        f"   |   Prophet  MAE: {mae:.2f} | RMSE: {rmse:.2f}</sup>"
    ),
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", y=1.08, x=0),
)
fig_t2.show()

Best CV MAE (T2): 20.859
Best params (T2): {'n_estimators': 196, 'learning_rate': 0.10513540620583425, 'num_leaves': 42, 'min_child_samples': 22, 'subsample': 0.6703060391028302}

Technique 2 — Prophet + LightGBM
  Test MAE : 27.43  (Prophet alone: 27.05)
  Test RMSE: 31.12  (Prophet alone: 29.75)


In [169]:
# ── Technique 3: Multi-level Prophet ensemble + LightGBM ────────────────
# Train extra Prophet models on weekly-resampled data (mean & median)
# Their predictions become additional features alongside daily Prophet features


def train_resampled_prophet(df_train_daily, freq, agg_fn, best_params):
    df_res = (
        df_train_daily.set_index("ds")
        .resample(freq)["y"]
        .agg(agg_fn)
        .reset_index()
        .dropna()
    )
    m = Prophet(
        weekly_seasonality=True,
        daily_seasonality=False,
        yearly_seasonality=False,
        **best_params,
    )
    m.add_country_holidays(country_name="FR")
    m.fit(df_res)
    return m


def get_prophet_features(model, ds_series, prefix):
    future = pd.DataFrame({"ds": pd.to_datetime(ds_series)})
    pred = model.predict(future)
    feats = pred[PROPHET_FEATS].copy()
    feats.columns = [f"{prefix}_{c}" for c in feats.columns]
    return feats.reset_index(drop=True)


# Train weekly Prophet models (mean and median aggregation)
print("Training weekly-mean Prophet...")
model_w_mean = train_resampled_prophet(df_train, "W", "mean", best_params)
print("Training weekly-median Prophet...")
model_w_median = train_resampled_prophet(df_train, "W", "median", best_params)


# ── Build feature sets ────────────────────────────────────────────────────
def build_features_t3(ds_series):
    ds_idx = pd.DatetimeIndex(pd.to_datetime(ds_series))

    # Daily Prophet features
    daily_pred = best_model.predict(pd.DataFrame({"ds": ds_idx}))
    feats = daily_pred[PROPHET_FEATS].copy()
    feats.columns = [f"daily_{c}" for c in feats.columns]
    feats = feats.reset_index(drop=True)

    # Weekly Prophet features
    feats_wm = get_prophet_features(model_w_mean, ds_idx, "w_mean")
    feats_wmd = get_prophet_features(model_w_median, ds_idx, "w_median")

    # Calendar — use DatetimeIndex directly (no .dt needed)
    cal = pd.DataFrame(
        {
            "day_of_week": ds_idx.dayofweek,
            "is_weekend": ds_idx.dayofweek.isin([5, 6]).astype(int),
        }
    ).reset_index(drop=True)

    return pd.concat([feats, feats_wm, feats_wmd, cal], axis=1)


X_t3_train = build_features_t3(df_train["ds"].values)
X_t3_test = build_features_t3(df_test["ds"].values)
print(f"Total features: {X_t3_train.shape[1]}")


# ── Optuna tuning ─────────────────────────────────────────────────────────
def objective_t3(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 30),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv.split(X_t3_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_t3_train.iloc[tr_idx], y_t2_train[tr_idx])
        fold_maes.append(
            mean_absolute_error(
                y_t2_train[val_idx], m.predict(X_t3_train.iloc[val_idx])
            )
        )
    return np.mean(fold_maes)


study_t3 = optuna.create_study(direction="minimize")
study_t3.optimize(objective_t3, n_trials=60)
print("Best CV MAE (T3):", round(study_t3.best_value, 3))

# ── Refit and evaluate ────────────────────────────────────────────────────
best_t3 = lgb.LGBMRegressor(**study_t3.best_params, verbosity=-1, random_state=42)
best_t3.fit(X_t3_train, y_t2_train)
t3_preds = best_t3.predict(X_t3_test)

t3_mae = mean_absolute_error(y_t2_test, t3_preds)
t3_rmse = root_mean_squared_error(y_t2_test, t3_preds)
print(f"\nTechnique 3 — Multi-level Prophet + LightGBM")
print(f"  Test MAE : {t3_mae:.2f}  (T2: {t2_mae:.2f}, Prophet: {mae:.2f})")
print(f"  Test RMSE: {t3_rmse:.2f}  (T2: {t2_rmse:.2f}, Prophet: {rmse:.2f})")

# ── Comparison plot (all 3 techniques) ────────────────────────────────────
fig_t3 = go.Figure()
fig_t3.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.5,
    )
)
fig_t3.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=y_t2_test,
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
    )
)
fig_t3.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["yhat"],
        name="T1: Prophet",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dot"),
        marker=dict(size=6),
    )
)
fig_t3.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t2_preds,
        name="T2: Prophet + LGBM",
        mode="lines+markers",
        line=dict(color="darkorange", width=2, dash="dash"),
        marker=dict(size=6),
    )
)
fig_t3.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t3_preds,
        name="T3: Multi-level + LGBM",
        mode="lines+markers",
        line=dict(color="mediumseagreen", width=2, dash="dashdot"),
        marker=dict(size=6),
    )
)
fig_t3.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig_t3.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train / Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig_t3.update_layout(
    title=(
        f"All Techniques Comparison — {selected_customer}<br>"
        f"<sup>"
        f"T1 Prophet  MAE: {mae:.2f} | RMSE: {rmse:.2f}   |   "
        f"T2 MAE: {t2_mae:.2f} | RMSE: {t2_rmse:.2f}   |   "
        f"T3 MAE: {t3_mae:.2f} | RMSE: {t3_rmse:.2f}"
        f"</sup>"
    ),
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", y=1.08, x=0),
)
fig_t3.show()

Training weekly-mean Prophet...


12:24:49 - cmdstanpy - INFO - Chain [1] start processing


12:24:49 - cmdstanpy - INFO - Chain [1] done processing
12:24:49 - cmdstanpy - INFO - Chain [1] start processing
12:24:49 - cmdstanpy - INFO - Chain [1] done processing


Training weekly-median Prophet...
Total features: 20
Best CV MAE (T3): 20.404

Technique 3 — Multi-level Prophet + LightGBM
  Test MAE : 25.59  (T2: 27.43, Prophet: 27.05)
  Test RMSE: 29.12  (T2: 31.12, Prophet: 29.75)


In [170]:
# ── Technique 2 (log1p): same Prophet features, log-transformed target ──
# log1p handles zero-sale days safely: log1p(0)=0, expm1 reverses it

y_log_train = np.log1p(y_t2_train)
y_log_test = np.log1p(y_t2_test)  # only used for reference, not evaluation


# Optuna — CV in log space (consistent with training objective)
def objective_t2_log(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 30),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv.split(X_t2_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_t2_train.iloc[tr_idx], y_log_train[tr_idx])
        preds_log = m.predict(X_t2_train.iloc[val_idx])
        # evaluate in original space for a fair fold MAE
        fold_maes.append(
            mean_absolute_error(np.expm1(y_log_train[val_idx]), np.expm1(preds_log))
        )
    return np.mean(fold_maes)


study_t2_log = optuna.create_study(direction="minimize")
study_t2_log.optimize(objective_t2_log, n_trials=60)
print("Best CV MAE (T2-log):", round(study_t2_log.best_value, 3))

# Refit on full train
best_t2_log = lgb.LGBMRegressor(
    **study_t2_log.best_params, verbosity=-1, random_state=42
)
best_t2_log.fit(X_t2_train, y_log_train)

# Predict → inverse transform → evaluate in original space
t2_log_preds = np.expm1(best_t2_log.predict(X_t2_test))
t2_log_mae = mean_absolute_error(y_t2_test, t2_log_preds)
t2_log_rmse = root_mean_squared_error(y_t2_test, t2_log_preds)

print(f"\nTechnique 2 log1p — Prophet features + LightGBM")
print(f"  Test MAE : {t2_log_mae:.2f}  (T2 no-log: {t2_mae:.2f}, Prophet: {mae:.2f})")
print(
    f"  Test RMSE: {t2_log_rmse:.2f}  (T2 no-log: {t2_rmse:.2f}, Prophet: {rmse:.2f})"
)

fig_t2_log = go.Figure()
fig_t2_log.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.5,
    )
)
fig_t2_log.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=y_t2_test,
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
    )
)
fig_t2_log.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t2_preds,
        name="T2 (no log)",
        mode="lines+markers",
        line=dict(color="darkorange", width=2, dash="dash"),
        marker=dict(size=6),
    )
)
fig_t2_log.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t2_log_preds,
        name="T2 (log1p)",
        mode="lines+markers",
        line=dict(color="purple", width=2, dash="dash"),
        marker=dict(size=6),
    )
)
fig_t2_log.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["yhat"],
        name="T1: Prophet",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dot"),
        marker=dict(size=6),
    )
)
fig_t2_log.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig_t2_log.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train / Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig_t2_log.update_layout(
    title=(
        f"Technique 2: log1p vs no-log — {selected_customer}<br>"
        f"<sup>"
        f"T2 no-log  MAE: {t2_mae:.2f} | RMSE: {t2_rmse:.2f}   |   "
        f"T2 log1p  MAE: {t2_log_mae:.2f} | RMSE: {t2_log_rmse:.2f}"
        f"</sup>"
    ),
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", y=1.08, x=0),
)
fig_t2_log.show()

Best CV MAE (T2-log): 21.838

Technique 2 log1p — Prophet features + LightGBM
  Test MAE : 26.55  (T2 no-log: 27.43, Prophet: 27.05)
  Test RMSE: 30.16  (T2 no-log: 31.12, Prophet: 29.75)


In [171]:
# ── Technique 3 (log1p): multi-level Prophet features, log-transformed target
import holidays as pyholidays


def objective_t3_log(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 30),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv.split(X_t3_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_t3_train.iloc[tr_idx], y_log_train[tr_idx])
        preds_log = m.predict(X_t3_train.iloc[val_idx])
        fold_maes.append(
            mean_absolute_error(np.expm1(y_log_train[val_idx]), np.expm1(preds_log))
        )
    return np.mean(fold_maes)


study_t3_log = optuna.create_study(direction="minimize")
study_t3_log.optimize(objective_t3_log, n_trials=60)
print("Best CV MAE (T3-log):", round(study_t3_log.best_value, 3))

best_t3_log = lgb.LGBMRegressor(
    **study_t3_log.best_params, verbosity=-1, random_state=42
)
best_t3_log.fit(X_t3_train, y_log_train)

t3_log_preds = np.expm1(best_t3_log.predict(X_t3_test))
t3_log_mae = mean_absolute_error(y_t2_test, t3_log_preds)
t3_log_rmse = root_mean_squared_error(y_t2_test, t3_log_preds)
print(f"T3 log1p  MAE: {t3_log_mae:.2f} | RMSE: {t3_log_rmse:.2f}")

# ── Calendar regressors ───────────────────────────────────────────────────
# French holiday dates as sorted ordinals for fast searchsorted lookup
_years = range(df_train["ds"].dt.year.min(), df_test["ds"].dt.year.max() + 2)
_fr_hols = pyholidays.France(years=_years)
_hol_ords = np.array(sorted(d.toordinal() for d in _fr_hols.keys()))


def add_calendar_regressors(ds_series):
    ds_idx = pd.DatetimeIndex(pd.to_datetime(ds_series))
    date_ords = np.array([d.toordinal() for d in ds_idx.date])

    # days to next holiday (cap at 30 if none ahead)
    idx_next = np.searchsorted(_hol_ords, date_ords)
    days_to_next = np.where(
        idx_next < len(_hol_ords),
        _hol_ords[np.minimum(idx_next, len(_hol_ords) - 1)] - date_ords,
        30,
    )

    # days since last holiday (cap at 30 if none before)
    idx_last = np.searchsorted(_hol_ords, date_ords, side="right") - 1
    days_from_last = np.where(
        idx_last >= 0,
        date_ords - _hol_ords[np.maximum(idx_last, 0)],
        30,
    )

    return pd.DataFrame(
        {
            "days_to_next_holiday": days_to_next,
            "days_from_last_holiday": days_from_last,
            "is_month_start": (ds_idx.day <= 5).astype(int),
            "is_month_end": (ds_idx.day >= 26).astype(int),
        }
    ).reset_index(drop=True)


cal_train = add_calendar_regressors(df_train["ds"].values)
cal_test = add_calendar_regressors(df_test["ds"].values)

X_t3_enh_train = pd.concat([X_t3_train.reset_index(drop=True), cal_train], axis=1)
X_t3_enh_test = pd.concat([X_t3_test.reset_index(drop=True), cal_test], axis=1)
print(f"\nEnhanced features: {X_t3_enh_train.shape[1]}  (was {X_t3_train.shape[1]})")


# ── T3 log1p + calendar regressors ───────────────────────────────────────
def objective_t3_cal(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 30),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv.split(X_t3_enh_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_t3_enh_train.iloc[tr_idx], y_log_train[tr_idx])
        preds_log = m.predict(X_t3_enh_train.iloc[val_idx])
        fold_maes.append(
            mean_absolute_error(np.expm1(y_log_train[val_idx]), np.expm1(preds_log))
        )
    return np.mean(fold_maes)


study_t3_cal = optuna.create_study(direction="minimize")
study_t3_cal.optimize(objective_t3_cal, n_trials=60)
print("Best CV MAE (T3-log+cal):", round(study_t3_cal.best_value, 3))

best_t3_cal = lgb.LGBMRegressor(
    **study_t3_cal.best_params, verbosity=-1, random_state=42
)
best_t3_cal.fit(X_t3_enh_train, y_log_train)

t3_cal_preds = np.expm1(best_t3_cal.predict(X_t3_enh_test))
t3_cal_mae = mean_absolute_error(y_t2_test, t3_cal_preds)
t3_cal_rmse = root_mean_squared_error(y_t2_test, t3_cal_preds)
print(f"T3 log1p + calendar  MAE: {t3_cal_mae:.2f} | RMSE: {t3_cal_rmse:.2f}")

# Feature importance
feat_imp = pd.Series(
    best_t3_cal.feature_importances_,
    index=X_t3_enh_train.columns,
).sort_values(ascending=False)
print("\nTop 10 features:")
print(feat_imp.head(10).to_string())

# ── Final summary ─────────────────────────────────────────────────────────
summary = {
    "T1: Prophet": (mae, rmse),
    "T2: P+LGBM": (t2_mae, t2_rmse),
    "T2: P+LGBM (log1p)": (t2_log_mae, t2_log_rmse),
    "T3: Multi-lvl+LGBM": (t3_mae, t3_rmse),
    "T3: Multi-lvl (log1p)": (t3_log_mae, t3_log_rmse),
    "T3: log1p + Calendar": (t3_cal_mae, t3_cal_rmse),
}
print("\n── Summary ──────────────────────────────────────")
print(f"{'Model':<28} {'MAE':>8} {'RMSE':>8}")
print("-" * 46)
for name, (m_mae, m_rmse) in summary.items():
    marker = "  ◀ best" if m_mae == min(v[0] for v in summary.values()) else ""
    print(f"{name:<28} {m_mae:>8.2f} {m_rmse:>8.2f}{marker}")

# ── Comparison chart (all 6) ──────────────────────────────────────────────
colors = ["crimson", "darkorange", "purple", "mediumseagreen", "teal", "royalblue"]
dashes = ["dot", "dash", "dash", "dashdot", "dashdot", "solid"]
preds_map = [
    (best_eval["ds"], best_eval["yhat"], "T1: Prophet"),
    (df_test["ds"], t2_preds, "T2: P+LGBM"),
    (df_test["ds"], t2_log_preds, "T2: P+LGBM (log1p)"),
    (df_test["ds"], t3_preds, "T3: Multi-lvl+LGBM"),
    (df_test["ds"], t3_log_preds, "T3: Multi-lvl (log1p)"),
    (df_test["ds"], t3_cal_preds, "T3: log1p + Calendar"),
]

fig_all = go.Figure()
fig_all.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.4,
    )
)
fig_all.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=y_t2_test,
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=7),
    )
)
for (xs, ys, label), color, dash in zip(preds_map, colors, dashes):
    m_mae, _ = summary[label]
    fig_all.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            name=f"{label}  MAE={m_mae:.1f}",
            mode="lines+markers",
            line=dict(color=color, width=2, dash=dash),
            marker=dict(size=5),
        )
    )
fig_all.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig_all.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train / Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig_all.update_layout(
    title=f"All Models Comparison — {selected_customer}",
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=550,
    legend=dict(orientation="h", y=1.14, x=0),
)
fig_all.show()

Best CV MAE (T3-log): 21.121
T3 log1p  MAE: 25.16 | RMSE: 27.19

Enhanced features: 24  (was 20)
Best CV MAE (T3-log+cal): 21.349
T3 log1p + calendar  MAE: 25.14 | RMSE: 27.64

Top 10 features:
daily_trend            310
daily_yhat_upper       172
w_median_yhat_lower    164
daily_yhat             144
w_median_yhat_upper    115
w_mean_yhat_lower      103
w_mean_weekly          101
w_median_weekly         86
daily_weekly            70
w_mean_yhat_upper       67

── Summary ──────────────────────────────────────
Model                             MAE     RMSE
----------------------------------------------
T1: Prophet                     27.05    29.75
T2: P+LGBM                      27.43    31.12
T2: P+LGBM (log1p)              26.55    30.16
T3: Multi-lvl+LGBM              25.59    29.12
T3: Multi-lvl (log1p)           25.16    27.19
T3: log1p + Calendar            25.14    27.64  ◀ best


In [172]:
# ── Residual Model: Prophet baseline + LightGBM correction ───────────────
# Prophet predicts trend + seasonality → yhat
# LightGBM learns to predict what Prophet gets wrong (the residual)
# Final prediction = prophet_yhat + lgbm_correction

# In-sample Prophet predictions on train set
prophet_train_full = best_model.predict(df_train[["ds"]])
prophet_test_full = best_model.predict(df_test[["ds"]])

train_residuals = df_train["y"].values - prophet_train_full["yhat"].values
print(
    f"Residuals — mean: {train_residuals.mean():.2f} | std: {train_residuals.std():.2f}"
)
print(f"Range: [{train_residuals.min():.1f}, {train_residuals.max():.1f}]")

# Use multi-level Prophet + calendar features (X_t3_enh) to explain residuals
X_res_train = X_t3_enh_train.reset_index(drop=True).copy()
X_res_test = X_t3_enh_test.reset_index(drop=True).copy()
y_res_train = train_residuals

# ── Optuna ────────────────────────────────────────────────────────────────
_prophet_yhat_train = prophet_train_full["yhat"].values
_y_train_vals = df_train["y"].values


def objective_resid(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 30),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv.split(X_res_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_res_train.iloc[tr_idx], y_res_train[tr_idx])
        correction = m.predict(X_res_train.iloc[val_idx])
        final_preds = _prophet_yhat_train[val_idx] + correction
        fold_maes.append(mean_absolute_error(_y_train_vals[val_idx], final_preds))
    return np.mean(fold_maes)


study_resid = optuna.create_study(direction="minimize")
study_resid.optimize(objective_resid, n_trials=60)
print("Best CV MAE (residual):", round(study_resid.best_value, 3))

# ── Refit + evaluate ──────────────────────────────────────────────────────
best_resid = lgb.LGBMRegressor(**study_resid.best_params, verbosity=-1, random_state=42)
best_resid.fit(X_res_train, y_res_train)

correction_test = best_resid.predict(X_res_test)
resid_preds = prophet_test_full["yhat"].values + correction_test
resid_mae = mean_absolute_error(y_t2_test, resid_preds)
resid_rmse = root_mean_squared_error(y_t2_test, resid_preds)

print(f"\nResidual model       MAE: {resid_mae:.2f} | RMSE: {resid_rmse:.2f}")
print(f"T3 log1p + Calendar  MAE: {t3_cal_mae:.2f} | RMSE: {t3_cal_rmse:.2f}")
print(f"Prophet alone        MAE: {mae:.2f} | RMSE: {rmse:.2f}")

# ── Feature importance ────────────────────────────────────────────────────
resid_imp = pd.Series(
    best_resid.feature_importances_, index=X_res_train.columns
).sort_values(ascending=False)
print("\nTop 10 features (residual model):")
print(resid_imp.head(10).to_string())

# ── Plot ──────────────────────────────────────────────────────────────────
fig_resid = go.Figure()
fig_resid.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.4,
    )
)
fig_resid.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=y_t2_test,
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=7),
    )
)
fig_resid.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["yhat"],
        name=f"Prophet  MAE={mae:.1f}",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dot"),
        marker=dict(size=5),
    )
)
fig_resid.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t3_cal_preds,
        name=f"T3 log1p+Cal  MAE={t3_cal_mae:.1f}",
        mode="lines+markers",
        line=dict(color="royalblue", width=2, dash="dash"),
        marker=dict(size=5),
    )
)
fig_resid.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=resid_preds,
        name=f"Residual model  MAE={resid_mae:.1f}",
        mode="lines+markers",
        line=dict(color="darkorange", width=2),
        marker=dict(size=6),
    )
)
fig_resid.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig_resid.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train / Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig_resid.update_layout(
    title=(
        f"Residual Model (Prophet + LightGBM correction) — {selected_customer}<br>"
        f"<sup>Residual  MAE: {resid_mae:.2f} | RMSE: {resid_rmse:.2f}"
        f"   |   Prophet  MAE: {mae:.2f} | RMSE: {rmse:.2f}</sup>"
    ),
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", y=1.08, x=0),
)
fig_resid.show()

Residuals — mean: -0.00 | std: 23.34
Range: [-69.7, 70.4]
Best CV MAE (residual): 17.783

Residual model       MAE: 26.87 | RMSE: 29.43
T3 log1p + Calendar  MAE: 25.14 | RMSE: 27.64
Prophet alone        MAE: 27.05 | RMSE: 29.75

Top 10 features (residual model):
daily_trend             380
w_median_yhat_lower      99
daily_yhat_upper         62
w_mean_yhat_lower        54
days_to_next_holiday     39
w_mean_weekly            16
daily_weekly             11
w_median_weekly          11
w_median_yhat_upper       4
w_mean_yhat_upper         1


In [173]:
# ── T3 log1p + Calendar + Lag features ───────────────────────────────────
# Add lag_7, lag_14, rolling_mean (shift=7 to avoid test leakage) from actual sales
# Computed on the full timeline so test lags draw from the training tail

df_full = (
    pd.concat([df_train[["ds", "y"]], df_test[["ds", "y"]]])
    .sort_values("ds")
    .reset_index(drop=True)
)

# lag_7 / lag_14: safe for 7-day horizon — test values always pull from train
df_full["lag_7"] = df_full["y"].shift(7)
df_full["lag_14"] = df_full["y"].shift(14)

# rolling_mean: use shift(7) so the window always ends 7+ days before current date
# → no leakage even for the last test day
df_full["rolling_mean_7"] = df_full["y"].shift(7).rolling(7).mean()
df_full["rolling_mean_14"] = df_full["y"].shift(7).rolling(14).mean()
df_full["rolling_std_7"] = df_full["y"].shift(7).rolling(7).std().fillna(0)

LAG_COLS = ["lag_7", "lag_14", "rolling_mean_7", "rolling_mean_14", "rolling_std_7"]

train_mask = df_full["ds"].isin(df_train["ds"])
test_mask = df_full["ds"].isin(df_test["ds"])

lag_train = df_full[train_mask][LAG_COLS].reset_index(drop=True)
lag_test = df_full[test_mask][LAG_COLS].reset_index(drop=True)

# Drop train rows where any lag is NaN (first 14+7 = 21 rows)
valid_mask = lag_train.notna().all(axis=1)
print(f"Train rows after dropping NaN lags: {valid_mask.sum()} / {len(lag_train)}")
print(f"NaN in lag_test: {lag_test.isna().sum().sum()}")

X_lag_train = pd.concat(
    [X_t3_enh_train.reset_index(drop=True)[valid_mask], lag_train[valid_mask]],
    axis=1,
)
y_lag_train = y_log_train[valid_mask.values]

X_lag_test = pd.concat(
    [X_t3_enh_test.reset_index(drop=True), lag_test],
    axis=1,
)
print(f"Total features: {X_lag_train.shape[1]}")

# ── Optuna ────────────────────────────────────────────────────────────────
tscv_lag = TimeSeriesSplit(n_splits=3)  # fresh splitter for the shorter train set


def objective_lag(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 3, 20),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        verbosity=-1,
        random_state=42,
    )
    fold_maes = []
    for tr_idx, val_idx in tscv_lag.split(X_lag_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_lag_train.iloc[tr_idx], y_lag_train[tr_idx])
        preds_log = m.predict(X_lag_train.iloc[val_idx])
        fold_maes.append(
            mean_absolute_error(np.expm1(y_lag_train[val_idx]), np.expm1(preds_log))
        )
    return np.mean(fold_maes)


study_lag = optuna.create_study(direction="minimize")
study_lag.optimize(objective_lag, n_trials=60)
print("Best CV MAE (lag model):", round(study_lag.best_value, 3))

# ── Refit + evaluate ──────────────────────────────────────────────────────
best_lag = lgb.LGBMRegressor(**study_lag.best_params, verbosity=-1, random_state=42)
best_lag.fit(X_lag_train, y_lag_train)

lag_preds = np.expm1(best_lag.predict(X_lag_test))
lag_mae = mean_absolute_error(y_t2_test, lag_preds)
lag_rmse = root_mean_squared_error(y_t2_test, lag_preds)

print(f"\nT3 log1p + Cal + Lags  MAE: {lag_mae:.2f} | RMSE: {lag_rmse:.2f}")
print(f"T3 log1p + Calendar    MAE: {t3_cal_mae:.2f} | RMSE: {t3_cal_rmse:.2f}")
print(f"Residual model         MAE: {resid_mae:.2f} | RMSE: {resid_rmse:.2f}")
print(f"Prophet alone          MAE: {mae:.2f} | RMSE: {rmse:.2f}")

# ── Feature importance ────────────────────────────────────────────────────
lag_imp = pd.Series(
    best_lag.feature_importances_, index=X_lag_train.columns
).sort_values(ascending=False)
print("\nTop 15 features (lag model):")
print(lag_imp.head(15).to_string())

# ── Full summary ──────────────────────────────────────────────────────────
summary_final = {
    "T1: Prophet": (mae, rmse),
    "T2: P+LGBM (log1p)": (t2_log_mae, t2_log_rmse),
    "T3: Multi-lvl (log1p)": (t3_log_mae, t3_log_rmse),
    "T3: log1p + Calendar": (t3_cal_mae, t3_cal_rmse),
    "Residual (Prophet+LGBM)": (resid_mae, resid_rmse),
    "T3: log1p + Cal + Lags": (lag_mae, lag_rmse),
}
best_mae_final = min(v[0] for v in summary_final.values())
print("\n── Final Summary ─────────────────────────────────────")
print(f"{'Model':<30} {'MAE':>8} {'RMSE':>8}")
print("-" * 48)
for name, (m_mae, m_rmse) in summary_final.items():
    marker = "  ◀ best" if m_mae == best_mae_final else ""
    print(f"{name:<30} {m_mae:>8.2f} {m_rmse:>8.2f}{marker}")

# ── Plot ──────────────────────────────────────────────────────────────────
fig_lag = go.Figure()
fig_lag.add_trace(
    go.Scatter(
        x=df_train["ds"],
        y=df_train["y"],
        name="Train",
        mode="lines",
        line=dict(color="steelblue", width=1.5),
        opacity=0.4,
    )
)
fig_lag.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=y_t2_test,
        name="Actual",
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=7),
    )
)
fig_lag.add_trace(
    go.Scatter(
        x=best_eval["ds"],
        y=best_eval["yhat"],
        name=f"Prophet  MAE={mae:.1f}",
        mode="lines+markers",
        line=dict(color="crimson", width=2, dash="dot"),
        marker=dict(size=5),
    )
)
fig_lag.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=t3_cal_preds,
        name=f"T3 log1p+Cal  MAE={t3_cal_mae:.1f}",
        mode="lines+markers",
        line=dict(color="royalblue", width=2, dash="dash"),
        marker=dict(size=5),
    )
)
fig_lag.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=resid_preds,
        name=f"Residual  MAE={resid_mae:.1f}",
        mode="lines+markers",
        line=dict(color="darkorange", width=2, dash="dashdot"),
        marker=dict(size=5),
    )
)
fig_lag.add_trace(
    go.Scatter(
        x=df_test["ds"],
        y=lag_preds,
        name=f"T3+Cal+Lags  MAE={lag_mae:.1f}",
        mode="lines+markers",
        line=dict(color="mediumseagreen", width=2),
        marker=dict(size=6),
    )
)
fig_lag.add_shape(
    type="line",
    x0=split_date,
    x1=split_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="gray", dash="dot", width=1.5),
)
fig_lag.add_annotation(
    x=split_date,
    y=1,
    xref="x",
    yref="paper",
    text="Train / Test split",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(color="gray", size=11),
)
fig_lag.update_layout(
    title=(
        f"T3 log1p + Calendar + Lag Features — {selected_customer}<br>"
        f"<sup>Lags  MAE: {lag_mae:.2f} | RMSE: {lag_rmse:.2f}"
        f"   |   Residual  MAE: {resid_mae:.2f}"
        f"   |   Prophet  MAE: {mae:.2f}</sup>"
    ),
    xaxis_title="Date",
    yaxis_title="Sales Quantity",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", y=1.08, x=0),
)
fig_lag.show()

Train rows after dropping NaN lags: 61 / 81
NaN in lag_test: 0
Total features: 29
Best CV MAE (lag model): 23.096

T3 log1p + Cal + Lags  MAE: 22.33 | RMSE: 24.48
T3 log1p + Calendar    MAE: 25.14 | RMSE: 27.64
Residual model         MAE: 26.87 | RMSE: 29.43
Prophet alone          MAE: 27.05 | RMSE: 29.75

Top 15 features (lag model):
daily_yhat_upper        273
rolling_mean_14         255
lag_14                  192
rolling_mean_7          181
daily_yhat              159
rolling_std_7           154
w_mean_weekly            93
daily_yhat_lower         59
lag_7                    53
w_median_weekly          47
daily_trend              47
w_median_yhat_lower      39
days_to_next_holiday     30
w_mean_yhat_lower        25
daily_weekly             20

── Final Summary ─────────────────────────────────────
Model                               MAE     RMSE
------------------------------------------------
T1: Prophet                       27.05    29.75
T2: P+LGBM (log1p)                26.55 

## LightGBM 